In [5]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, make_scorer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [ ]:
# LOAD DATA
import gdown
import os

DATA_DIR = "../data/processed/"
DRIVE_FOLDER_ID = "1ooZ-wuu7CWji4f6C_TG-JuwO5o97_lzh"

os.makedirs(DATA_DIR, exist_ok=True)

drive_datasets = [
    "adasyn_balanced.parquet",
    "smote_balanced.parquet",
    "equal_undersampled.parquet",
    "moderate_undersampled.parquet",
    "NearMiss_equal.parquet",
    "NearMiss_moderate.parquet",
    "tomek_links.parquet",
]

missing = [f for f in drive_datasets if not os.path.exists(os.path.join(DATA_DIR, f))]

if missing:
    print(f"Descargando carpeta de Drive ({len(missing)} archivos faltantes)...")
    gdown.download_folder(
        id=DRIVE_FOLDER_ID,
        output=DATA_DIR,
        quiet=False,
        remaining_ok=True,
    )
else:
    print("Todos los archivos ya existen localmente.")

adasyn_data                = pd.read_parquet(f"{DATA_DIR}adasyn_balanced.parquet")
smote_data                 = pd.read_parquet(f"{DATA_DIR}smote_balanced.parquet")
equal_undersampled_data    = pd.read_parquet(f"{DATA_DIR}equal_undersampled.parquet")
moderate_undersampled_data = pd.read_parquet(f"{DATA_DIR}moderate_undersampled.parquet")
NearMiss_equal_data        = pd.read_parquet(f"{DATA_DIR}NearMiss_equal.parquet")
NearMiss_moderate_data     = pd.read_parquet(f"{DATA_DIR}NearMiss_moderate.parquet")
tomek_links_data           = pd.read_parquet(f"{DATA_DIR}tomek_links.parquet")

original_data = pd.read_parquet("../data/splits/train_original.parquet")

Todos los archivos ya existen localmente.


In [3]:
selected_features = [
                    'Elevation', 'Aspect', 'Horizontal_Distance_To_Hydrology',
                    'Vertical_Distance_To_Hydrology', 'Horizontal_Distance_To_Roadways', 'Hillshade_9am',
                    'Hillshade_Noon', 'Horizontal_Distance_To_Fire_Points', 'Wilderness_Area1', 'Wilderness_Area2', 
                    'Wilderness_Area3', 'Wilderness_Area4', 'Soil_Type2', 'Soil_Type4', 'Soil_Type10', 
                    'Soil_Type12', 'Soil_Type22', 'Soil_Type23', 'Soil_Type38', 'Soil_Type39', 'Soil_Type40', 
                    'Euclidean_Distance_To_Hydrology', 'Distance_To_Hydrology_To_Roadways_Ratio', 'Distance_To_Fire_To_Hydrology_Ratio', 
                    'Total_Distance', 'Hydrology_Slope', 'Aspect_North_South', 'Cover_Type'
                    ] 



# Dataset Selection

In this section we will choose between the balanced datasets. We will train a RandomForest with same hyperparameters for each of the datasets and test the performance so that we can find the dataset with the best results.

In [4]:
def split_features_and_target(df, selected_features):
    df = df[selected_features]
    print(df.shape)
    X = df.drop('Cover_Type', axis=1)
    y = df['Cover_Type']
    return X,y

In [5]:
# PARAMETROS
RANDOM_STATE = 42
N_SPLITS = 5

hyperparameters = {
    'n_estimators': 100,
    'max_depth': 15,
    'min_samples_split': 5,
    'min_samples_leaf': 2,
    'random_state': RANDOM_STATE
}

print(f"Hiperparámetros a usar: {hyperparameters}\n")

Hiperparámetros a usar: {'n_estimators': 100, 'max_depth': 15, 'min_samples_split': 5, 'min_samples_leaf': 2, 'random_state': 42}



In [6]:
datasets = {
    'SMOTE': smote_data,
    'ADASYN': adasyn_data,
    'Random_Undersampling_Moderate': moderate_undersampled_data,
    'Random_Undersampling_Equal': equal_undersampled_data,
    'NearMiss_Moderate': NearMiss_moderate_data,
    'NearMiss_Equal': NearMiss_equal_data,
    'Tomek_Links': tomek_links_data,
}

In [16]:
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    'Accuracy': make_scorer(accuracy_score),
    'Precision': make_scorer(precision_score, average='macro', zero_division=0),
    'Recall': make_scorer(recall_score, average='macro', zero_division=0),
    'F1': make_scorer(f1_score, average='macro', zero_division=0)
}

results = {}
for dataset_name, data in datasets.items():
    print(f"\n{'='*70}")
    print(f"Processing: {dataset_name}")
    print(f"{'='*70}")
    
    # Separar X, y
    X, y = split_features_and_target(data, selected_features=selected_features)
    
    # Crear modelo
    rf_model = RandomForestClassifier(**hyperparameters, n_jobs=-1)
    
    # Pipeline con scaler
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', rf_model)
    ])
    
    # Cross-Validation
    print(f"Executing {N_SPLITS}-Fold Cross Validation...")
    cv_results = cross_validate(pipeline, X, y, cv=cv, scoring=scoring, return_train_score=True)
    
    # Calcular promedios
    test_acc = cv_results['test_Accuracy'].mean()
    test_acc_std = cv_results['test_Accuracy'].std()
    test_f1 = cv_results['test_F1'].mean()
    test_f1_std = cv_results['test_F1'].std()
    test_precision = cv_results['test_Precision'].mean()
    test_recall = cv_results['test_Recall'].mean()
    
    train_acc = cv_results['train_Accuracy'].mean()
    
    # Guardar resultados
    results[dataset_name] = {
        'Train Accuracy': train_acc,
        'Test Accuracy': test_acc,
        'Test Accuracy Std': test_acc_std,
        'Test Precision': test_precision,
        'Test Recall': test_recall,
        'Test F1': test_f1,
        'Test F1 Std': test_f1_std,
        'Samples': len(X)
    }


Processing: SMOTE
(1586221, 28)
Executing 5-Fold Cross Validation...

Processing: ADASYN
(1596800, 28)
Executing 5-Fold Cross Validation...

Processing: Random_Undersampling_Moderate
(83665, 28)
Executing 5-Fold Cross Validation...

Processing: Random_Undersampling_Equal
(15386, 28)
Executing 5-Fold Cross Validation...

Processing: NearMiss_Moderate
(83665, 28)
Executing 5-Fold Cross Validation...

Processing: NearMiss_Equal
(15386, 28)
Executing 5-Fold Cross Validation...

Processing: Tomek_Links
(449014, 28)
Executing 5-Fold Cross Validation...


In [17]:
original_data.replace([float('inf'), float('-inf')], pd.NA, inplace=True)
original_data = original_data.dropna()
X,y = split_features_and_target(original_data, selected_features)
rf_model = RandomForestClassifier(**hyperparameters, n_jobs=-1)
pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', rf_model)
    ])

print(f"Executing {N_SPLITS}-Fold Cross Validation...")
    
cv_results = cross_validate(pipeline, X, y, cv=cv, scoring=scoring, return_train_score=True)
    
    # Calcular promedios
test_acc = cv_results['test_Accuracy'].mean()
test_acc_std = cv_results['test_Accuracy'].std()
test_f1 = cv_results['test_F1'].mean()
test_f1_std = cv_results['test_F1'].std()
test_precision = cv_results['test_Precision'].mean()
test_recall = cv_results['test_Recall'].mean()
    
train_acc = cv_results['train_Accuracy'].mean()

(464710, 28)
Executing 5-Fold Cross Validation...


In [18]:
print("ORIGINAL DATASET RESULTS")
print(f"Test Accuracy: {test_acc}")
print(f"Test acc_std: {test_acc_std}")
print(f"Test f1: {test_f1}")
print(f"Test f1_std: {test_f1_std}")
print(f"Test Precision: {test_precision}")
print(f"Test Recall: {test_recall}")

ORIGINAL DATASET RESULTS
Test Accuracy: 0.8499472789481611
Test acc_std: 0.0024862397125504578
Test f1: 0.7869800917544166
Test f1_std: 0.0023058468898746716
Test Precision: 0.8980481006852464
Test Recall: 0.7368950077904909


In [19]:
print("DATASET COMPARISON")
print(f"{'='*70}\n")

results_df = pd.DataFrame(results).T
print(results_df)

# Resumen
print(f"\n{'='*70}")
print("BEST DATASETS")
print(f"{'='*70}")

best_acc = results_df['Test Accuracy'].idxmax()
best_f1 = results_df['Test F1'].idxmax()

print(f"\Best Accuracy: {best_acc}")
print(f"   Accuracy: {results_df.loc[best_acc, 'Test Accuracy']:.4f}")
print(f"   F1-Score: {results_df.loc[best_acc, 'Test F1']:.4f}")

print(f"\Best F1-Score: {best_f1}")
print(f"   Accuracy: {results_df.loc[best_f1, 'Test Accuracy']:.4f}")
print(f"   F1-Score: {results_df.loc[best_f1, 'Test F1']:.4f}")

# Ordenar por Accuracy descendente
print(f"\nRanking Test Accuracy:")
ranking = results_df[['Test Accuracy', 'Test F1']].sort_values('Test Accuracy', ascending=False)
for idx, (dataset, row) in enumerate(ranking.iterrows(), 1):
    print(f"  {idx}. {dataset:30s} - Accuracy: {row['Test Accuracy']:.4f}, F1: {row['Test F1']:.4f}")

DATASET COMPARISON

                               Train Accuracy  Test Accuracy  \
SMOTE                                0.931505       0.925710   
ADASYN                               0.919103       0.911722   
Random_Undersampling_Moderate        0.924532       0.877392   
Random_Undersampling_Equal           0.959395       0.849084   
NearMiss_Moderate                    0.959147       0.931883   
NearMiss_Equal                       0.982240       0.941701   
Tomek_Links                          0.870600       0.856612   

                               Test Accuracy Std  Test Precision  Test Recall  \
SMOTE                                   0.001399        0.925222     0.925710   
ADASYN                                  0.001419        0.912046     0.912239   
Random_Undersampling_Moderate           0.002404        0.878794     0.877528   
Random_Undersampling_Equal              0.006440        0.846245     0.849089   
NearMiss_Moderate                       0.002821        0.9299

In [20]:
results_df.to_csv("../results/dataset_results.csv")